Drug_API.ipynb

This notebook is only for demonstrating:

The native PyTorch Geometric API (Data, GATConv)
The wrapper layer you wrote (Drug_utils.py) with a clean forward pass
It uses tiny dummy graphs so it runs fast (no long training).

In [ ]:
# Imports
import torch
import torch.nn as nn

# Native PyG API (graph objects + GAT)
from torch_geometric.data import Data
from torch_geometric.nn import GATConv, global_mean_pool

# Metrics (used for a tiny demo later)
from sklearn.metrics import roc_auc_score

# Your wrapper layer (make sure Drug_utils.py is in the same folder)
# If this import fails, fix your Python env / file naming first.
from Drug_utils import TrainConfig, DrugGATEncoder, DrugInteractionModel

1) Transform a molecule into a graph (toy example)

In the real project you will build graphs from atom + bond information (often from SMILES via RDKit), but here we demonstrate the structure:

x: node features (one row per atom)
edge_index: edges (two rows; source nodes and destination nodes)

In [ ]:
# A tiny "molecule" with 3 atoms (toy example)
# Atom features could be: [atomic_number, degree, aromatic, ...]
x = torch.tensor([
    [6.0, 4.0, 0.0],   # carbon-like
    [8.0, 2.0, 0.0],   # oxygen-like
    [7.0, 3.0, 0.0],   # nitrogen-like
], dtype=torch.float)

# Bonds (undirected graph) encoded as two directed edges each
# 0--1 and 1--2
edge_index = torch.tensor([
    [0, 1, 1, 2],
    [1, 0, 2, 1],
], dtype=torch.long)

drug_graph = Data(x=x, edge_index=edge_index)
drug_graph

2) Native API demo: run a single GATConv forward

This shows you are using the native PyG classes directly.

In [ ]:
in_dim = drug_graph.num_node_features
hidden_dim = 8
heads = 2

gat = GATConv(in_channels=in_dim, out_channels=hidden_dim, heads=heads, concat=True)

# Forward on node features
h = gat(drug_graph.x, drug_graph.edge_index)
h.shape

3) Wrapper API demo: DrugInteractionModel forward pass

encode Drug A graph -> embedding vector
encode Drug B graph -> embedding vector
combine embeddings -> interaction logit

In [ ]:
# Create a second dummy drug graph (different atom features, different edges)
x2 = torch.tensor([
    [6.0, 3.0, 0.0],
    [6.0, 4.0, 1.0],
    [8.0, 1.0, 0.0],
    [7.0, 2.0, 0.0],
], dtype=torch.float)

edge_index2 = torch.tensor([
    [0, 1, 1, 2, 2, 3],
    [1, 0, 2, 1, 3, 2],
], dtype=torch.long)

drug_graph_2 = Data(x=x2, edge_index=edge_index2)

# Build your wrapper model
emb_dim = 16
encoder = DrugGATEncoder(in_dim=3, hidden_dim=8, out_dim=emb_dim, heads=2)
model = DrugInteractionModel(encoder=encoder, emb_dim=emb_dim)

# IMPORTANT: for a true *pair* model, you typically batch graphs with DataLoader.
# Here, we just do a single forward pass on two graphs.
logit = model(drug_graph, drug_graph_2)
logit

4) Tiny end-to-end demo (fast): predict interaction for a few synthetic pairs + ROC-AUC

This is not your full training notebook. It is just a minimal proof that:

- we can create drug graphs
- we can compute logits
- we can compute ROC-AUC

For the actual project, do full training/evaluation in Drug_example.ipynb.

In [ ]:
# Helper: make a few synthetic pairs (drugA, drugB, label)
# Label here is fake, only to demonstrate ROC-AUC calculation.
pairs = [
    (drug_graph, drug_graph_2, 1),
    (drug_graph, drug_graph, 0),
    (drug_graph_2, drug_graph_2, 0),
    (drug_graph_2, drug_graph, 1),
]

model.eval()
y_true = []
y_score = []

with torch.no_grad():
    for a, b, y in pairs:
        logit = model(a, b)
        prob = torch.sigmoid(logit).item()
        y_true.append(y)
        y_score.append(prob)

print('y_true :', y_true)
print('y_score:', [round(v, 4) for v in y_score])
print('ROC-AUC:', roc_auc_score(y_true, y_score))

5) Feature importance (minimal demonstration)

For molecular graphs, “feature importance” can mean:

- which atom features matter (atomic number, aromaticity, etc.)
- which atoms/bonds matter (substructure importance)

A simple baseline approach is perturbation importance:

- change one node-feature column
- measure how much the predicted probability changes

Below is a toy perturbation demo for a single pair. (For your real report, do this on many samples and summarize.)

In [ ]:
def perturb_feature_column(graph: Data, col_idx: int, delta: float = 1.0) -> Data:
    g = Data(x=graph.x.clone(), edge_index=graph.edge_index.clone())
    g.x[:, col_idx] = g.x[:, col_idx] + delta
    return g

model.eval()

base_prob = torch.sigmoid(model(drug_graph, drug_graph_2)).item()
print("Base prob:", round(base_prob, 6))

for col in range(drug_graph.num_node_features):
    perturbed = perturb_feature_column(drug_graph, col_idx=col, delta=1.0)
    p = torch.sigmoid(model(perturbed, drug_graph_2)).item()
    print(f"Column {col} prob:", round(p, 6), " | abs change:", round(abs(p - base_prob), 6))

What is covered in Drug_API.ipynb:

- Graph representation (Data(x, edge_index))
- Native PyG GATConv forward
- Wrapper model forward (DrugInteractionModel)
- Quick ROC-AUC calculation on a tiny synthetic set
- Minimal perturbation-based feature importance example